In [74]:
#Import packages for calculation
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import pynucastro as pyna

In [163]:
class MainphaseStar:
    #Stellar evolution simulator that opperates on the core conceit that the star remains in the Main Phase (where the primary energy in the star comes from hydrogen fusion) and remain in thermal and hydrostatic equilibrium


    def __init__(self, temp, dens, pres, lumin, pcenthydrogen, radius):
        #Takes 4 1d-arrays of equal size and 1 float.64
        self.t = temp #in kelvin
        self.d = dens #in gram/meters**3
        self.p = pres 
        self.L = lumin
        self.pcenthydrogen = pcenthydrogen #percent by mass of hydrogen for each layer
        self.r = radius #in meters
        #obtains the number of cells as well as the size of each cell 
        self.n = len(temp)
        self.rinc = self.r / self.n
        
        #use pynucastro to load reaction rates for p-p chain and c-n-o chain reactions, assuming the rates are bounded by the slowest reaction rate 
        rl = pyna.ReacLibLibrary()
        self.ppchainrate = rl.get_rate_by_name("p(p,)d")
        self.cnochainrate = rl.get_rate_by_name("n14(p,g)o15")

        #radiation constant as defined by Stefan Bolzman Law
        self.a = 7.5657 * 10 **(-16) * 6.241509*10**12
        #units MeV (m**-3) (K**-4)

        #gravitational constant
        self.G = 6.6743*10**-13
        #units m**3 /(g*s**2)

    def rincrinitialize(self):
        #reinitializes the calculation for the radius increment of each cell
        self.rinc = self.r / self.n
        self.rbycell = (self.n - np.asarray(range(0, self.n)))*self.rinc

    def massbyrad(self):
        #initializes mass array for each cell, where mass is the volume * the density
        self.mass = np.zeros_like(self.t) #Mass of interior of star from each cell
        self.layermass = np.zeros_like(self.t) #mass of material in the layer composed by each cell
        for i in range(1, self.n + 1):
            self.mass[-i] = 4 / 3 * np.pi * (self.rinc * i) ** 3 * (np.sum(self.d[-i:])/i)  
        for i in range(0, self.n -1 ):
            self.layermass[i] = self.mass[i] - self.mass[i+1]
        self.layermass[-1] = self.mass[-1]

    def fusionenergy(self, density, perhydro, temp):
        #calculate fusion energy per unit mass based on temp and density of hydrogen- using p-p and c-n-o fusion paths 
        #Note: pynucastro assumes rho density is in grams/cm**3 thus we have to convert   
        
        if perhydro == 0 or temp == 0 or density == 0:
           return 0
        #evaluates reaction rates per cm^3 
        ppeval = self.ppchainrate[0].eval(temp, rho = (density*perhydro/(10**6)))
        cnoeval = self.cnochainrate.eval(temp, rho = (density*perhydro/(10**6)))

        #convert to reaction rates per unit mass
        ppeval = ppeval*10**6/density
        cnoeval = cnoeval*10**6/density

        #evaluate energy result per unit mass then converts mega electron volt to electron volt
        energyperunitmass = (ppeval*26.72 + cnoeval*25)*1000000 
        return energyperunitmass
    
    def drdm(self, m, r, rho):   
        #mass conservation equation
        if rho == 0:
            return 0
        
        return 1/(4*np.pi*r**2 *rho)

    def dpdm(self, m, p, r):
        #hydrostatic equilibrium
        return -self.G*m/(4*np.pi**4*r)
    
    def dLdm(self, m, L, nuc, r1, r2, rho):
        #energy conservation
        
        grav = 16/5*np.pi*self.G*rho**2*(r1**5 - r2**5)/m
        return nuc + grav 
    
    def dTdm(self, m, T, r, P, L, dens):
        #energy transport
        if P == 0 or r == 0 or P == 0 or dens == 0 or L == 0:
            return 0
         
        k = 10**17*dens* np.sign(T) * (np.abs(T)) **(7/2)
        grad = ((3*k*L*P)/(16*np.pi*self.a*self.G*m*T**4))
        return -(self.G*m*T)/(4*np.pi*r**4*P)*grad 

        
    def massolve(self, masses):
        #Takes list of t masses to solve system at. First mass is assumed to be mass of original system 
        #returns a t by n array for each system quantity of values at each radius interval n for that mass
        #evaluates from center out for each time
        
        #initialize arrays
        self.massbyrad()
        self.rincrinitialize()
        temp = np.zeros((len(masses), self.n)) 
        dens = np.zeros((len(masses), self.n)) 
        pres = np.zeros((len(masses), self.n)) 
        radis = np.zeros((len(masses), self.n)) 
        pcent = np.zeros((len(masses), self.n)) 
        luminos = np.zeros((len(masses), self.n))
        efromrad = np.zeros((len(masses), self.n)) 
        intmass = np.zeros((len(masses), self.n))
        laymass = np.zeros((len(masses), self.n))
        temp[0, :] = self.t
        dens[0, :] = self.d
        pres[0, :] = self.p
        luminos[0, :] = self.L
        radis[0, :] = self.rbycell
        pcent[0, :] = self.pcenthydrogen
        intmass[0, :] = self.mass
        laymass[0, :] = self.layermass

        for q in range(1, len(masses)):

            #calculate the mass difference between this stage and previous mass
            massdif = masses[q] - masses[q-1]

            #evaluate per unit fusion energy at each cell and convert that into difference in mass across layers
            for i in range(0, self.n):
                efromrad[q-1, -i] = self.fusionenergy( float(dens[q-1, -i]), float(pcent[q-1, -i]), float(temp[q-1, -i]))
            pcentfusion = efromrad[q-1, :] / np.sum(efromrad[q-1, :])
            massdifbycell = massdif * pcentfusion
            laymass[q, :] = laymass[q-1, :] + massdifbycell

            #update hydrogen percentages
            pcent[q, :] = (pcent[q-1, :] * laymass[q-1, :] + massdifbycell) / laymass[q-1, :]

            #solve for radiuses
            for i in range(0, self.n):
                rho = sum(dens[q-1, i:])/(self.n - i)
                output = sp.integrate.solve_ivp(self.drdm, (masses[q-1], masses[q]), (radis[q-1, i],), args = (rho,))
                if i > 0:    
                    if output["y"][-1][-1] < radis[q, i -1]:
                        radis[q, i] = output["y"][-1][-1]
                else:
                    radis[q, i] = output["y"][-1][-1]

            #solve for preassures 
            for i in range(0, self.n):
                output = sp.integrate.solve_ivp(self.dpdm, (masses[q-1], masses[q]), (pres[q-1, i],), args = (radis[q, i],))
                pres[q, i] = output["y"][-1][-1]
            
            #solve for luminosities
            for i in range(0, self.n):
                rho = sum(dens[q-1, i:])/(self.n - i)
                output = sp.integrate.solve_ivp(self.dLdm, (masses[q-1], masses[q]), (luminos[q-1, i],), args = (efromrad[q-1, i], radis[q-1, i], radis[q, i], rho))
                luminos[q, i] = output["y"][-1][-1]
            
            #solve for temperatures
            for i in range(0, self.n):
                output = sp.integrate.solve_ivp(self.dTdm, (masses[q-1], masses[q]), (temp[q-1, i],), args = (radis[q, i], pres[q, i], luminos[q, i], dens[q-1, i]))
                temp[q, i] = output["y"][-1][-1]
                intmass[q, i] = sum(laymass[q, i:])

            #density wrong!   
            for i in range(0, self.n-1):
                dens[q, i] = laymass[q, i]/(4/3*np.pi*radis[q, i]**3- 4/3*np.pi*radis[q, i+1]**3)
            

        return (temp, dens, pres, luminos, pcent, radis, intmass, laymass, efromrad)

In [169]:
#1 solar mass run from MESA simulation start point
temp = np.asarray(range(0, 150))/150*(10**7.1346983)
dens = np.append(np.logspace(-6.7, 1.8, 10), np.logspace(1.8, 1.891, 140))*1000
pres =   np.asarray(range(0, 150)) * (3.4*10**11)/150*101.325*1000
radis = 700000*1000
lumin = np.logspace(-99, -0.55634, 150)
pcent = np.ones(150)*0.698965
star = MainphaseStar(temp, dens, pres, lumin, pcent, radis)

In [ ]:
star.rincrinitialize()
star.massbyrad()
star.mass
masspoints = np.linspace(9.54475243*10**31, 9.54475243*10**31*0.9999965, 3)
output = star.massolve(masspoints)

C:\Users\Ramie\AppData\Local\Temp\ipykernel_27236\4123531610.py:86: RuntimeWarning: invalid value encountered in divide
  grad = ((3*k*L*P)/(16*np.pi*self.a*self.G*m*T**4))
C:\Users\Ramie\AppData\Local\Temp\ipykernel_27236\4123531610.py:86: RuntimeWarning: overflow encountered in multiply
  grad = ((3*k*L*P)/(16*np.pi*self.a*self.G*m*T**4))
C:\Users\Ramie\AppData\Local\Temp\ipykernel_27236\4123531610.py:72: RuntimeWarning: divide by zero encountered in scalar divide
  return -self.G*m/(4*np.pi**4*r)


[1.99526231e-04 1.75567629e-03 1.54485915e-02 1.35935639e-01
 1.19612833e+00 1.05250029e+01 9.26118728e+01 8.14912747e+02
 7.17060097e+03 6.30957344e+04 6.30957344e+04 6.31909197e+04
 6.32862485e+04 6.33817212e+04 6.34773379e+04 6.35730988e+04
 6.36690042e+04 6.37650543e+04 6.38612493e+04 6.39575894e+04
 6.40540748e+04 6.41507058e+04 6.42474826e+04 6.43444053e+04
 6.44414743e+04 6.45386897e+04 6.46360518e+04 6.47335607e+04
 6.48312168e+04 6.49290201e+04 6.50269711e+04 6.51250697e+04
 6.52233164e+04 6.53217113e+04 6.54202546e+04 6.55189466e+04
 6.56177875e+04 6.57167775e+04 6.58159168e+04 6.59152057e+04
 6.60146443e+04 6.61142330e+04 6.62139719e+04 6.63138613e+04
 6.64139014e+04 6.65140923e+04 6.66144345e+04 6.67149280e+04
 6.68155731e+04 6.69163700e+04 6.70173190e+04 6.71184203e+04
 6.72196741e+04 6.73210807e+04 6.74226403e+04 6.75243530e+04
 6.76262192e+04 6.77282391e+04 6.78304129e+04 6.79327408e+04
 6.80352231e+04 6.81378599e+04 6.82406517e+04 6.83435985e+04
 6.84467006e+04 6.854995

C:\Users\Ramie\AppData\Local\Temp\ipykernel_27236\4123531610.py:86: RuntimeWarning: overflow encountered in power
  grad = ((3*k*L*P)/(16*np.pi*self.a*self.G*m*T**4))
